# Cấp 3 — MMKG Ẩm thực Việt Nam: truy vấn ảnh có giải thích bằng đồ thị tri thức

Notebook này làm phần **dữ liệu tự xây** của cấp 3 (`Project03_Multimodal_XAI.pdf`):

1. Dựng một MMKG nhỏ tiếng Việt về ẩm thực: các món, nguyên liệu, nhóm nguyên liệu, cách chế biến, loại món, tỉnh và vùng. Mỗi ảnh nối với món nó mô tả bằng quan hệ `mô_tả`.
2. Chạy phương pháp tốt nhất của cấp 1–2 (CLIP + điểm đường đi trên đồ thị + GNN + khớp truy vấn↔node, trộn bằng α, λ chọn trên val), điều chỉnh cho MMKG.
3. Đo truy vấn trực tiếp và gián tiếp; chạy ablation (bỏ thuộc tính KG, KG ngẫu nhiên, xáo liên kết ảnh–món); đo fidelity, validity, sparsity của giải thích.
4. Phân tích khả năng chuyển giao: CLIP đa ngữ so với CLIP tiếng Anh, truy vấn tiếng Việt / không dấu / tiếng Anh, so với Flickr8k và Action Genome.

Nguồn dữ liệu, cách tạo triple và điều kiện sử dụng nằm trong `mmkg/DATA_CARD.md`.

In [ ]:
import os, sys, json, re, math, random, time, subprocess, unicodedata, hashlib, itertools, warnings
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MMKG_SMOKE", "0") == "1"   # chạy thử bằng đặc trưng giả, KHÔNG phải kết quả
SEEDS = [0, 1, 2]
ON_KAGGLE = os.path.exists("/kaggle")
OUT = os.environ.get("MMKG_OUT", "/kaggle/working/mmkg_out" if ON_KAGGLE else "./mmkg_out")
os.makedirs(OUT, exist_ok=True)

def find_mmkg_dir():
    cands = [os.environ.get("MMKG_DIR", ""), "mmkg", ".", "/kaggle/working/du-lieu-do-thi/mmkg"]
    for c in cands:
        if c and os.path.exists(os.path.join(c, "kg_data.py")) and os.path.exists(os.path.join(c, "images.csv")):
            return os.path.abspath(c)
    # lấy dữ liệu từ GitHub (repo công khai), chỉ thư mục mmkg/
    dst = "/kaggle/working/du-lieu-do-thi" if ON_KAGGLE else "./du-lieu-do-thi"
    if not os.path.exists(dst):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", "-b", "kaggle-notebook-v11",
                        "https://github.com/trinhphamtrunghieu/du-lieu-do-thi", dst], check=True)
        subprocess.run(["git", "-C", dst, "sparse-checkout", "set", "mmkg"], check=True)
    return os.path.abspath(os.path.join(dst, "mmkg"))

MMKG_DIR = find_mmkg_dir()
sys.path.insert(0, MMKG_DIR)
import kg_data
print("MMKG_DIR =", MMKG_DIR, "| OUT =", OUT, "| SMOKE =", SMOKE)

## 1. Dựng MMKG

Thực thể và triple tri thức sinh từ `kg_data.py` (nhóm biên soạn). Ảnh lấy từ Wikimedia Commons; mỗi ảnh có một triple `ảnh —mô_tả→ món`. Ảnh là node của đồ thị nhưng không tính vào số **thực thể**: đề bài tách thực thể (50–150) và ảnh (200–500).

In [ ]:
ents, kg_triples, slug = kg_data.build_kg()
ENT = {e["id"]: e for e in ents}
DISHES = [e["id"] for e in ents if e["kind"] == "món"]
DIDX = {d: i for i, d in enumerate(DISHES)}
imgs = pd.read_csv(os.path.join(MMKG_DIR, "images.csv"))
imgs["path"] = imgs["file"].map(lambda f: os.path.join(MMKG_DIR, f))
assert imgs["path"].map(os.path.exists).all(), "thiếu file ảnh"
img_triples = [(r.image_id, kg_data.REL["depicts"], r.dish_id) for r in imgs.itertuples()]
all_triples = kg_triples + img_triples

ent_df = pd.DataFrame(ents)
ent_df["name_en"] = [kg_data.EN_NAME.get(i[5:], "") if k == "món" else kg_data.EN_ENTITY.get(n, n)
                     for i, k, n in zip(ent_df.id, ent_df.kind, ent_df.name)]
tri_df = pd.DataFrame(all_triples, columns=["head", "relation", "tail"])
tri_df["source"] = np.where(tri_df.relation == kg_data.REL["depicts"], "Wikimedia Commons (liên kết ảnh–món)", "kg_data.py (nhóm biên soạn)")
ent_df.to_csv(os.path.join(OUT, "entities.csv"), index=False)
tri_df.to_csv(os.path.join(OUT, "triples.csv"), index=False)

print("Thực thể theo loại:", dict(Counter(e["kind"] for e in ents)))
print("Triple theo quan hệ:", dict(Counter(r for _, r, _ in all_triples)))
print("Ảnh theo giấy phép:", dict(Counter(imgs.license)))
print("Ảnh theo cách liên kết:", dict(Counter(imgs.link_method)))
n_dish_img = imgs.dish_id.nunique()
checks = [("Số thực thể trong [50, 150]", len(ents), 50 <= len(ents) <= 150),
          ("Số triple trong [300, 1000]", len(all_triples), 300 <= len(all_triples) <= 1000),
          ("Số ảnh trong [200, 500]", len(imgs), 200 <= len(imgs) <= 500)]
print()
for name, v, ok in checks:
    print(f"{'ĐẠT' if ok else 'CHƯA'}  {name}: {v}")
print(f"Món có ảnh: {n_dish_img}/{len(DISHES)}; ảnh/món: min {imgs.dish_id.value_counts().min()}, "
      f"trung vị {int(imgs.dish_id.value_counts().median())}, max {imgs.dish_id.value_counts().max()}")
print("Món không có ảnh:", [ENT[d]["name"] for d in DISHES if d not in set(imgs.dish_id)])

### Cấu trúc truy vết của KG

Ma trận `R1[món, thực thể]` = 1 nếu có triple trực tiếp món → thực thể (loại, xuất xứ, cách chế biến, nguyên liệu). `R2` = 1 nếu đi qua 2 bước: món → nguyên liệu → nhóm, hoặc món → tỉnh → vùng. Đây là các "đường đi" dùng để chấm điểm và để giải thích.

In [ ]:
EIDX = {e["id"]: i for i, e in enumerate(ents)}
ATTR_RELS = [kg_data.REL[k] for k in ("type", "origin", "method", "ingredient")]
LINK_RELS = [kg_data.REL["group"], kg_data.REL["region"]]

def reach_mats(triples):
    """R1, R2 và bảng đường đi (để giải thích) từ danh sách triple tri thức."""
    R1 = np.zeros((len(DISHES), len(ents)), np.float32)
    R2 = np.zeros_like(R1)
    up = defaultdict(list)                       # nguyên liệu → nhóm, tỉnh → vùng
    for h, r, t in triples:
        if r in LINK_RELS:
            up[h].append((r, t))
    path2 = defaultdict(list)                    # (món, đích 2 bước) → [(r1, trung gian, r2)]
    for h, r, t in triples:
        if h in DIDX and r in ATTR_RELS:
            R1[DIDX[h], EIDX[t]] = 1
            for r2, t2 in up.get(t, []):
                R2[DIDX[h], EIDX[t2]] = 1
                path2[(h, t2)].append((r, t, r2))
    return R1, R2, path2

R1_true, R2_true, PATH2_true = reach_mats(kg_triples)
print("R1:", int(R1_true.sum()), "cặp món–thực thể 1 bước; R2:", int(R2_true.sum()), "cặp 2 bước")

## 2. Chia dữ liệu

Chia theo **ảnh trong từng món** (train / val / test ≈ 50 / 20 / 30%), theo nhóm ảnh gần trùng (cùng tác giả + cùng gốc tên tệp) để ảnh gần giống nhau không rơi vào hai tập khác nhau. Cạnh `mô_tả` của ảnh val/test **không** có trong đồ thị: mô hình phải tự suy ra ảnh mô tả món nào. Ba seed chia khác nhau.

In [ ]:
def make_split(seed):
    rng = random.Random(1000 + seed)
    split = {}
    for d, g in imgs.groupby("dish_id"):
        groups = defaultdict(list)
        for r in g.itertuples():
            groups[r.dup_group].append(r.image_id)
        keys = sorted(groups); rng.shuffle(keys)
        n = len(g)
        n_test = 0 if n < 2 else max(1, round(0.3 * n))
        n_val = 0 if n < 3 else max(1, round(0.2 * n))
        cnt = {"test": 0, "val": 0}
        for k in keys:
            if cnt["test"] < n_test and n - cnt["test"] - len(groups[k]) >= 1:
                s = "test"
            elif cnt["val"] < n_val and n - cnt["test"] - cnt["val"] - len(groups[k]) >= 1:
                s = "val"
            else:
                s = "train"
            if s != "train":
                cnt[s] += len(groups[k])
            for i in groups[k]:
                split[i] = s
    return split

SPLITS = {s: make_split(s) for s in SEEDS}
for s in SEEDS:
    print("seed", s, dict(Counter(SPLITS[s].values())))
json.dump(SPLITS, open(os.path.join(OUT, "splits.json"), "w"), indent=0)
IMG_IDS = list(imgs.image_id)
IIDX = {i: k for k, i in enumerate(IMG_IDS)}
IMG_DISH = np.array([DIDX[d] for d in imgs.dish_id])

## 3. Bộ truy vấn

- **Trực tiếp:** tên món, ví dụ "Bánh xèo".
- **Gián tiếp:** không nêu tên món, chỉ nêu tổ hợp thuộc tính đủ để xác định món (loại, xuất xứ, vùng, cách chế biến, nguyên liệu, nhóm nguyên liệu), ví dụ "bánh mặn của Huế, có tôm". Một số thuộc tính chỉ suy ra được qua 2 bước (vùng qua tỉnh, nhóm qua nguyên liệu).

Truy vấn sinh tự động từ KG thật. Bộ **val** và bộ **test** dùng cách diễn đạt khác nhau và tổ hợp thuộc tính khác nhau; chỉ bộ val dùng để chọn tham số. Mỗi truy vấn được đánh giá trên ảnh của tập tương ứng; đáp án là mọi ảnh của các món thỏa mọi thuộc tính.

Hạn chế cần nói rõ: truy vấn gián tiếp sinh theo mẫu nên dùng đúng từ vựng của KG, liên kết thực thể gần như luôn đúng. Vì vậy có thêm bộ **diễn đạt tự nhiên** (24 câu viết tay, dùng từ ngoài KG như "heo", "miền Tây", "chả giò") để đo độ bền; bộ này chỉ dùng trên test.

In [ ]:
def dish_attrs(d):
    """Các thuộc tính của món, kèm số bước trong KG."""
    out = []
    for e in np.nonzero(R1_true[DIDX[d]])[0]:
        out.append((ents[e]["id"], 1))
    for e in np.nonzero(R2_true[DIDX[d]])[0]:
        out.append((ents[e]["id"], 2))
    return out

def satisfiers(cons):
    ok = np.ones(len(DISHES), bool)
    for e, _ in cons:
        ok &= (R1_true[:, EIDX[e]] + R2_true[:, EIDX[e]]) > 0
    return [DISHES[i] for i in np.nonzero(ok)[0]]

KIND_ORDER = {"loại món": 0, "nguyên liệu": 3, "nhóm nguyên liệu": 3, "cách chế biến": 2, "tỉnh": 1, "vùng": 1}

def render(cons, lang, style):
    parts = sorted(cons, key=lambda c: KIND_ORDER[ENT[c[0]]["kind"]])
    nm = (lambda e: ENT[e]["name"]) if lang == "vi" else (lambda e: kg_data.EN_ENTITY.get(ENT[e]["name"], ENT[e]["name"]))
    head = [c for c in parts if ENT[c[0]]["kind"] == "loại món"]
    rest = [c for c in parts if ENT[c[0]]["kind"] != "loại món"]
    if lang == "vi":
        h = nm(head[0][0]) if head else "món"
        bits = []
        for e, _ in rest:
            k = ENT[e]["kind"]
            if k in ("tỉnh",): bits.append(f"của {nm(e)}")
            elif k == "vùng": bits.append(f"của {nm(e)}")
            elif k == "cách chế biến": bits.append(f"được {nm(e)}")
            else: bits.append(f"có {nm(e)}")
        if style == "test":
            return h + " " + ", ".join(bits)
        return f"tìm ảnh {h} nào " + " và ".join(bits)
    else:
        h = nm(head[0][0]) if head else "dish"
        bits = []
        for e, _ in rest:
            k = ENT[e]["kind"]
            if k == "tỉnh": bits.append(f"from {slug(ENT[e]['name']).replace('_', ' ').title()}")
            elif k == "vùng": bits.append(f"from {nm(e)}")
            elif k == "cách chế biến": bits.append(nm(e))
            else: bits.append(f"with {nm(e)}")
        meth = [b for b, (e, _) in zip(bits, rest) if ENT[e]["kind"] == "cách chế biến"]
        other = [b for b, (e, _) in zip(bits, rest) if ENT[e]["kind"] != "cách chế biến"]
        return "a " + " ".join(meth + [h] + other)

NATURAL_GROUPS = {"hải sản", "lá gói", "rau thơm và lá"}   # nhóm nguyên liệu dùng được trong câu hỏi tự nhiên

def make_queries():
    rng_v, rng_t = random.Random(7), random.Random(11)
    Q = []
    for d in DISHES:
        if d not in set(imgs.dish_id):
            continue
        Q.append(dict(qid=f"dir_{d}", group="direct", gold=[d], cons=[], vi=ENT[d]["name"],
                      en=kg_data.EN_NAME[d[5:]], hops=0, construction="tên món"))
        attrs = dish_attrs(d)
        combos = []
        attrs = [(e, h) for e, h in attrs if ENT[e]["kind"] != "nhóm nguyên liệu" or ENT[e]["name"] in NATURAL_GROUPS]
        for k in (2, 3):
            for c in itertools.combinations(attrs, k):
                kinds = Counter(ENT[e]["kind"] for e, _ in c)
                if any(v > 1 and k != "nguyên liệu" for k, v in kinds.items()):
                    continue
                if (kinds["tỉnh"] and kinds["vùng"]) or (kinds["nguyên liệu"] and kinds["nhóm nguyên liệu"]):
                    continue
                g = satisfiers(c)
                if len(g) == 1:
                    combos.append(c)
        if len(combos) < 2:
            continue
        # test ưu tiên tổ hợp có loại món + thuộc tính 2 bước + 3 ràng buộc; val lấy ngẫu nhiên tổ hợp khác
        rng_t.shuffle(combos)
        has_type = lambda c: any(ENT[e]["kind"] == "loại món" for e, _ in c)
        combos.sort(key=lambda c: (-has_type(c), -max(h for _, h in c), -len(c)))
        ct = combos[0]
        cv = [c for c in combos[1:] if set(c) != set(ct)]
        cv = cv[rng_v.randrange(len(cv))] if cv else None
        for c, split_name in ((ct, "test"), (cv, "val")):
            if c is None:
                continue
            Q.append(dict(qid=f"ind_{split_name}_{d}", group="indirect", gold=[d], cons=[e for e, _ in c],
                          vi=render(c, "vi", split_name), en=render(c, "en", split_name),
                          hops=max(h for _, h in c), use=split_name,
                          construction="tổ hợp thuộc tính xác định duy nhất món, sinh từ KG"))
    return Q

QUERIES = make_queries()

# Bộ diễn đạt tự nhiên: viết tay (nhóm, có hỗ trợ AI), dùng từ đồng nghĩa / cách nói không có trong KG
# ("heo", "miền Tây", "chả giò"...). Chỉ dùng để đánh giá trên test, không dùng chọn tham số, không thêm bí danh cho chúng.
PARAPHRASE = [
    ("tô bún cay đặc sản xứ Huế có sả", ["bun_bo_hue"]),
    ("món nước của miền Tây nấu từ mắm", ["bun_mam", "lau_mam"]),
    ("bánh gói lá dong ngày Tết", ["banh_chung"]),
    ("bánh nếp hình trụ gói lá chuối của miền Nam", ["banh_tet"]),
    ("cà phê đánh trứng kiểu Hà Nội", ["ca_phe_trung"]),
    ("cà phê pha phin uống với đá và sữa", ["ca_phe_sua_da"]),
    ("bánh chiên giòn đổ trong khuôn nhỏ của Vũng Tàu", ["banh_khot"]),
    ("bánh tráng cuốn tôm thịt ăn sống, không chiên", ["goi_cuon"]),
    ("chả giò chiên giòn", ["nem_ran"]),
    ("cơm sườn nướng Sài Gòn", ["com_tam"]),
    ("bún ăn với đậu rán và mắm tôm", ["bun_dau"]),
    ("cá kho trong nồi đất kiểu Nam Bộ", ["ca_kho_to"]),
    ("thịt heo kho với trứng ngày Tết miền Nam", ["thit_kho"]),
    ("món canh nấu me với cá", ["canh_chua"]),
    ("mì sợi vàng của Quảng Nam ăn với bánh tráng", ["mi_quang"]),
    ("nem làm từ thịt heo lên men của Thanh Hóa", ["nem_chua"]),
    ("bò cuốn lá lốt nướng", ["bo_la_lot"]),
    ("bánh hấp mỏng nhân thịt và mộc nhĩ", ["banh_cuon"]),
    ("bánh bột năng trong, nhân tôm, gói lá chuối kiểu Huế", ["banh_bot_loc"]),
    ("bánh ngọt nhân đậu xanh và sầu riêng của miền Tây", ["banh_pia"]),
    ("phở nấu với thịt gà", ["pho_ga"]),
    ("bún với chả cá chiên, nước dùng có cà chua và dứa", ["bun_cha_ca"]),
    ("bánh xèo nhỏ nướng khuôn đất có trứng của Phan Rang", ["banh_can"]),
    ("bánh mì kẹp thịt và đồ chua", ["banh_mi"]),
]
for t, g in PARAPHRASE:
    QUERIES.append(dict(qid="para_" + g[0], group="paraphrase", gold=["dish_" + x for x in g], cons=[], vi=t,
                        hops=0, use="test", construction="viết tay, diễn đạt tự nhiên"))
for q in QUERIES:
    q["vi_nodiac"] = slug(q["vi"]).replace("_", " ")
    q.setdefault("use", "both")
json.dump(QUERIES, open(os.path.join(OUT, "queries.json"), "w"), ensure_ascii=False, indent=1)
print(Counter((q["group"], q["use"]) for q in QUERIES))
print("Truy vấn gián tiếp test cần 2 bước suy luận:", sum(q["hops"] == 2 for q in QUERIES if q["use"] == "test"))
for q in [q for q in QUERIES if q["use"] == "test"][:6]:
    print(f"  [{q['hops']} bước] {q['vi']!r}  |  {q['en']!r}  →  {ENT[q['gold'][0]]['name']}")

## 4. Trích đặc trưng CLIP (đóng băng)

Mô hình chính: **CLIP đa ngữ** `xlm-roberta-large-ViT-H-14` (LAION-5B), vì truy vấn là tiếng Việt. Để phân tích chuyển giao, chạy thêm CLIP đa ngữ nhỏ `xlm-roberta-base-ViT-B-32` và CLIP tiếng Anh `ViT-L-14` (OpenAI, backbone của phần A).

In [ ]:
MODELS = {"mclip_h14": ("xlm-roberta-large-ViT-H-14", "frozen_laion5b_s13b_b90k"),
          "mclip_b32": ("xlm-roberta-base-ViT-B-32", "laion5b_s13b_b90k"),
          "clip_l14_en": ("ViT-L-14", "openai")}
MAIN = "mclip_h14"

def dish_desc(d, lang, triples=None):
    """Mô tả văn bản của node món, sinh từ các triple của nó."""
    triples = kg_triples if triples is None else triples
    by = defaultdict(list)
    for h, r, t in triples:
        if h == d:
            by[r].append(ENT[t]["name"] if lang == "vi" else kg_data.EN_ENTITY.get(ENT[t]["name"], ENT[t]["name"]))
    R = kg_data.REL
    if lang == "vi":
        s = f"{ENT[d]['name']}, {', '.join(by[R['type']])}"
        if by[R["origin"]]: s += f", xuất xứ {', '.join(by[R['origin']])}"
        if by[R["method"]]: s += f", {' và '.join(by[R['method']])}"
        if by[R["ingredient"]]: s += f", gồm {', '.join(by[R['ingredient']])}"
    else:
        s = f"{kg_data.EN_NAME[d[5:]]}, {', '.join(by[R['type']])}"
        if by[R["origin"]]: s += f", from {', '.join(by[R['origin']])}"
        if by[R["method"]]: s += f", {' and '.join(by[R['method']])}"
        if by[R["ingredient"]]: s += f", with {', '.join(by[R['ingredient']])}"
    return s

def variant_triples(variant, seed):
    if variant in ("full", "shuffled_links"):
        return kg_triples
    if variant == "no_kg_attr":
        return [t for t in kg_triples if t[1] in LINK_RELS]
    if variant == "random_kg":
        rng = random.Random(500 + seed)
        out = [t for t in kg_triples if t[1] not in ATTR_RELS]
        for r in ATTR_RELS:
            ts = [t for t in kg_triples if t[1] == r]
            tails = [t[2] for t in ts]; rng.shuffle(tails)
            seen = set()
            for (h, _, _), t2 in zip(ts, tails):
                if (h, r, t2) not in seen:
                    seen.add((h, r, t2)); out.append((h, r, t2))
        return out
    raise ValueError(variant)

def all_texts():
    T = set()
    for q in QUERIES:
        T.update([q[f] for f in ("vi", "en", "vi_nodiac") if f in q])
    for e in ents:
        T.add(e["name"]); T.add(kg_data.EN_ENTITY.get(e["name"], e["name"]))
    for d in DISHES:
        T.add(dish_desc(d, "vi")); T.add(dish_desc(d, "en")); T.add(kg_data.EN_NAME[d[5:]])
    for s in SEEDS:                       # mô tả node của KG ngẫu nhiên (ablation)
        rt = variant_triples("random_kg", s)
        for d in DISHES:
            T.add(dish_desc(d, "vi", rt))
    return sorted(T)

TEXTS = all_texts()

def fake_feats(key):
    """Đặc trưng giả cho SMOKE: ảnh/tên cùng món gần nhau."""
    rng = np.random.default_rng(0)
    P = rng.normal(size=(len(DISHES), 64))
    def h(s):
        return np.random.default_rng(int(hashlib.md5(s.encode()).hexdigest()[:8], 16)).normal(size=64)
    I = np.stack([P[IMG_DISH[k]] + 1.2 * h(i) for k, i in enumerate(IMG_IDS)])
    T = []
    for s in TEXTS:
        v = h(s)
        for d in DISHES:
            if ENT[d]["name"].lower() in s.lower() or kg_data.EN_NAME[d[5:]].lower() in s.lower():
                v = v + 1.5 * P[DIDX[d]]
        T.append(v)
    T = np.stack(T)
    nz = lambda X: (X / np.linalg.norm(X, axis=1, keepdims=True)).astype(np.float32)
    return nz(I), nz(T)

def tokenize(tok, texts):
    """open_clip 3.2.0 gọi batch_encode_plus, đã bị bỏ ở transformers mới; gọi thẳng tokenizer HF khi cần."""
    try:
        return tok(texts)
    except AttributeError:
        return tok.tokenizer(list(texts), return_tensors="pt", max_length=tok.context_length,
                             padding="max_length", truncation=True).input_ids

def clip_feats(key):
    cache = os.path.join(OUT, f"feats_{key}.npz")
    if os.path.exists(cache) and not SMOKE:
        z = np.load(cache, allow_pickle=True)
        if list(z["texts"]) == TEXTS:
            return z["img"], z["txt"]
    if SMOKE:
        return fake_feats(key)
    import torch, open_clip
    from PIL import Image
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    arch, pt = MODELS[key]
    t0 = time.time()
    model, _, prep = open_clip.create_model_and_transforms(arch, pretrained=pt, device=dev)
    model.eval(); tok = open_clip.get_tokenizer(arch)
    I, T = [], []
    with torch.no_grad(), torch.autocast(device_type="cuda", enabled=(dev == "cuda")):
        for b in range(0, len(imgs), 32):
            x = torch.stack([prep(Image.open(p).convert("RGB")) for p in imgs.path[b:b + 32]]).to(dev)
            I.append(torch.nn.functional.normalize(model.encode_image(x).float(), dim=-1).cpu().numpy())
        for b in range(0, len(TEXTS), 128):
            t = tokenize(tok, TEXTS[b:b + 128]).to(dev)
            T.append(torch.nn.functional.normalize(model.encode_text(t).float(), dim=-1).cpu().numpy())
    I, T = np.concatenate(I), np.concatenate(T)
    np.savez(cache, img=I, txt=T, texts=np.array(TEXTS, dtype=object))
    print(f"{key}: {arch}/{pt} trên {dev}, {time.time() - t0:.0f}s, ảnh {I.shape}, văn bản {T.shape}")
    del model
    if dev == "cuda":
        torch.cuda.empty_cache()
    return I, T

FEATS = {}
for k in MODELS:
    try:
        FEATS[k] = clip_feats(k)
    except Exception as ex:
        if k == MAIN:
            raise
        print("Không tải được", k, "→", repr(ex)[:300])
if MAIN not in FEATS:
    MAIN = [k for k in ("mclip_b32", "clip_l14_en") if k in FEATS][0]
    print("Dùng mô hình chính thay thế:", MAIN)
TIDX = {t: i for i, t in enumerate(TEXTS)}
print("Mô hình có đặc trưng:", list(FEATS), "| chính:", MAIN)

## 5. Phương pháp: CLIP + đồ thị tri thức

Phương pháp tốt nhất của cấp 1–2 (phần A: CLIP + GNN + điểm đường đi + nhánh khớp truy vấn↔node, trộn bằng z-score) được giữ cấu trúc, điều chỉnh cho MMKG:

$$s(q, i) = \alpha\, z(\cos(q, i)) + (1-\alpha)\, z\Big(\sum_{d} P(d \mid q)\,\big[\lambda\, p_{GNN}(d \mid i) + (1-\lambda)\, p_{kNN}(d \mid i)\big]\Big)$$

- **Truy vấn → món** $P(d\mid q)$ (thay cho nhánh khớp truy vấn↔node): liên kết thực thể trong truy vấn với node KG (khớp chuỗi dài nhất); điểm đường đi = tỉ lệ ràng buộc mà món thỏa qua 1 bước (trọng số 1) hoặc 2 bước (trọng số $w_2$); cộng với độ tương đồng CLIP giữa truy vấn và mô tả node món sinh từ triple; softmax nhiệt độ $T$. Trộn bằng $\eta$.
- **Ảnh → món**: ảnh val/test chưa có cạnh `mô_tả`. $p_{kNN}$: bỏ phiếu theo ảnh train gần nhất (đặc trưng CLIP ảnh). $p_{GNN}$: GraphSAGE quan hệ (R-SAGE, 2 lớp) trên đồ thị gồm node ảnh, node thực thể và mọi triple KG; huấn luyện dự đoán cạnh ảnh–món bằng cách che ngẫu nhiên một phần cạnh `mô_tả` của ảnh train.
- **Mọi tham số** ($\alpha, \lambda, \eta, T, w_2$) chọn trên val, theo từng seed.

Không còn nhánh ridge của phần A vì MMKG không có caption để học ánh xạ truy vấn → node; liên kết thực thể và mô tả node đảm nhận vai trò đó.

In [ ]:
def norm_text(s, strip):
    s = unicodedata.normalize("NFC", s.lower())
    if strip:
        s = unicodedata.normalize("NFD", s.replace("đ", "d"))
        s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return " " + re.sub(r"[^\w]+", " ", s).strip() + " "

ALIASES_VI = {"TP. Hồ Chí Minh": ["sài gòn", "hồ chí minh", "tp hcm"], "Bắc Bộ": ["miền bắc"],
              "Trung Bộ": ["miền trung"], "Nam Bộ": ["miền nam"]}

def alias_table(lang, strip):
    tab = []
    for e in ents:
        names = [e["name"]] + ALIASES_VI.get(e["name"], []) if lang == "vi" else []
        if lang == "en":
            names = [kg_data.EN_NAME[e["id"][5:]] if e["kind"] == "món" else
                     (slug(e["name"]).replace("_", " ") if e["kind"] == "tỉnh" else kg_data.EN_ENTITY[e["name"]])]
            if e["kind"] == "món":
                names.append(" ".join(kg_data.EN_NAME[e["id"][5:]].split()[:2]))
        for n in names:
            tab.append((norm_text(n, strip or lang == "en"), e["id"]))
    tab.sort(key=lambda x: -len(x[0]))
    return tab

ALIAS = {(l, s): alias_table(l, s) for l in ("vi", "en") for s in (False, True)}

def link_entities(text, lang):
    has_diac = any(unicodedata.category(c) == "Mn" for c in unicodedata.normalize("NFD", text)) or "đ" in text.lower()
    strip = (lang == "en") or not has_diac
    s = norm_text(text, strip)
    found = []
    for a, e in ALIAS[(lang, strip)]:
        if a in s:
            found.append(e)
            s = s.replace(a, " # ")
    return found

# kiểm tra nhanh liên kết thực thể trên truy vấn test
ok = tot = 0
for q in QUERIES:
    if q["group"] == "indirect":
        got = set(link_entities(q["vi"], "vi")); tot += 1; ok += got == set(q["cons"])
print(f"Liên kết thực thể đúng hoàn toàn trên {ok}/{tot} truy vấn gián tiếp (tiếng Việt có dấu)")

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

def zrow(S):
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-8)

def softmax(X, T):
    X = X / T
    X = X - X.max(1, keepdims=True)
    E = np.exp(X)
    return E / E.sum(1, keepdims=True)

class RSAGE(nn.Module):
    """GraphSAGE quan hệ: mỗi loại cạnh (kể cả chiều ngược) có ma trận riêng, gộp trung bình."""
    def __init__(self, d_in, d_h, n_rel, layers=2, drop=0.2):
        super().__init__()
        self.inp = nn.Linear(d_in, d_h)
        self.selfs = nn.ModuleList([nn.Linear(d_h, d_h) for _ in range(layers)])
        self.rels = nn.ModuleList([nn.ModuleList([nn.Linear(d_h, d_h, bias=False) for _ in range(n_rel)]) for _ in range(layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_h) for _ in range(layers)])
        self.drop = drop
    def forward(self, x, edges):
        h = self.inp(x)
        for l in range(len(self.selfs)):
            m = self.selfs[l](h)
            for r, (src, dst) in enumerate(edges):
                if len(src) == 0:
                    continue
                msg = self.rels[l][r](h[src])
                agg = torch.zeros_like(h).index_add_(0, dst, msg)
                deg = torch.zeros(h.shape[0], 1).index_add_(0, dst, torch.ones(len(dst), 1)).clamp(min=1)
                m = m + agg / deg
            h = h + F.dropout(F.gelu(self.norms[l](m)), self.drop, self.training)
        return F.normalize(h, dim=-1)

def build_graph(img_feat, ent_feat, triples, train_ids, labels, k_img=5):
    """Node: [ảnh..., thực thể...]. Cạnh: triple KG (2 chiều), ảnh→ảnh train gần nhất, mô_tả (ảnh train)."""
    n_img = len(IMG_IDS)
    rel_names = sorted({r for _, r, _ in triples})
    E = []
    for r in rel_names:
        src = [n_img + EIDX[h] for h, rr, t in triples if rr == r]
        dst = [n_img + EIDX[t] for h, rr, t in triples if rr == r]
        E.append((src, dst)); E.append((dst, src))
    tr = np.array([IIDX[i] for i in train_ids])
    S = img_feat @ img_feat[tr].T
    for k in range(n_img):
        S[k, np.nonzero(tr == k)[0]] = -9   # không tự nối
    nb = np.argsort(-S, 1)[:, :k_img]
    src = list(tr[nb].ravel()); dst = list(np.repeat(np.arange(n_img), k_img))
    E.append((src, dst))
    X = np.concatenate([img_feat, ent_feat])
    return X, E, tr

def train_gnn(img_feat, ent_feat, triples, split, labels, seed, epochs=300, mask_frac=0.4):
    torch.manual_seed(seed); np.random.seed(seed)
    train_ids = [i for i in IMG_IDS if split[i] == "train"]
    val_ids = [i for i in IMG_IDS if split[i] == "val"]
    X, E, tr = build_graph(img_feat, ent_feat, triples, train_ids, labels)
    X = torch.tensor(X, dtype=torch.float32)
    n_img = len(IMG_IDS)
    dish_nodes = torch.tensor([n_img + EIDX[d] for d in DISHES])
    lab = torch.tensor(labels)
    def with_links(ids):
        idx = [IIDX[i] for i in ids]
        s = idx; d = [n_img + EIDX[DISHES[labels[k]]] for k in idx]
        return E + [(s, d), (d, s)]
    def to_t(E_):
        return [(torch.tensor(s, dtype=torch.long), torch.tensor(d, dtype=torch.long)) for s, d in E_]
    model = RSAGE(X.shape[1], 256, len(E) + 2)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    tau = 0.07
    full_E = to_t(with_links(train_ids))
    val_idx = torch.tensor([IIDX[i] for i in val_ids], dtype=torch.long)
    best, best_state, bad = -1, None, 0
    rng = np.random.default_rng(seed)
    for ep in range(epochs):
        model.train()
        perm = rng.permutation(len(train_ids))
        hide = [train_ids[k] for k in perm[: int(mask_frac * len(train_ids))]]
        keep = [train_ids[k] for k in perm[int(mask_frac * len(train_ids)):]]
        h = model(X, to_t(with_links(keep)))
        tgt = torch.tensor([IIDX[i] for i in hide])
        logits = h[tgt] @ h[dish_nodes].T / tau
        loss = F.cross_entropy(logits, lab[tgt])
        opt.zero_grad(); loss.backward(); opt.step()
        if ep % 10 == 9 and len(val_ids):
            model.eval()
            with torch.no_grad():
                h = model(X, full_E)
                acc = (h[val_idx] @ h[dish_nodes].T).argmax(1).eq(lab[val_idx]).float().mean().item()
            if acc > best:
                best, best_state, bad = acc, {k: v.clone() for k, v in model.state_dict().items()}, 0
            else:
                bad += 1
                if bad >= 8:
                    break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model, X, full_E, dish_nodes, tau, best

def gnn_probs(model, X, E_t, dish_nodes, tau):
    with torch.no_grad():
        h = model(X, E_t)
        return F.softmax(h[: len(IMG_IDS)] @ h[dish_nodes].T / tau, dim=1).numpy()

def knn_probs(img_feat, split, labels, k=10, temp=0.05):
    tr = np.array([IIDX[i] for i in IMG_IDS if split[i] == "train"])
    S = img_feat @ img_feat[tr].T
    P = np.zeros((len(IMG_IDS), len(DISHES)), np.float32)
    for n in range(len(IMG_IDS)):
        s = S[n].copy()
        s[tr == n] = -9
        nb = np.argsort(-s)[:k]
        w = np.exp((s[nb] - s[nb].max()) / temp)
        np.add.at(P[n], labels[tr[nb]], w)
        P[n] /= P[n].sum()
    return P

In [ ]:
def query_dish_scores(q_texts, lang, R1, R2, w2, desc_emb, txt_feat):
    """Điểm ràng buộc KG S_kg[q, d] và điểm văn bản S_txt[q, d]."""
    Skg = np.zeros((len(q_texts), len(DISHES)), np.float32)
    links = []
    for n, t in enumerate(q_texts):
        es = link_entities(t, lang)
        links.append(es)
        dish_hit = [e for e in es if e in DIDX]
        cons = [e for e in es if e not in DIDX]
        if cons:
            sat = np.stack([np.maximum(R1[:, EIDX[e]], w2 * R2[:, EIDX[e]]) for e in cons]).mean(0)
            Skg[n] = sat
        for e in dish_hit:
            Skg[n, DIDX[e]] = max(Skg[n, DIDX[e]], 1.0)
    qv = np.stack([txt_feat[TIDX[t]] for t in q_texts])
    Stxt = qv @ desc_emb.T
    return Skg, Stxt, links

def fast_mrr(S, G):
    """MRR nhanh; G: mặt nạ đáp án [truy vấn, gallery]."""
    best = np.where(G, S, -np.inf).max(1, keepdims=True)
    return float((1.0 / (1 + (S > best).sum(1))).mean())

def evaluate(S, gold_sets):
    """S: [truy vấn, ảnh trong gallery]; gold_sets: tập chỉ số ảnh đúng."""
    out = []
    for n, g in enumerate(gold_sets):
        order = np.argsort(-S[n], kind="stable")
        ranks = np.nonzero(np.isin(order, list(g)))[0] + 1
        r1 = ranks.min()
        dcg = sum(1 / math.log2(r + 1) for r in ranks if r <= 10)
        idcg = sum(1 / math.log2(r + 1) for r in range(1, min(len(g), 10) + 1))
        out.append(dict(R1=float(r1 <= 1), R5=float(r1 <= 5), R10=float(r1 <= 10), MRR=1 / r1, nDCG10=dcg / idcg))
    return pd.DataFrame(out)

### Một lần chạy = một (mô hình, ngôn ngữ truy vấn, biến thể KG, seed)

Hàm `run` chọn tham số trên val rồi báo cáo trên test. Các biến thể dùng cho ablation:
- `full`: KG thật.
- `no_kg_attr`: bỏ toàn bộ triple thuộc tính; chỉ còn tên món và liên kết ảnh–món.
- `random_kg`: xáo đuôi các triple thuộc tính giữa các món (giữ bậc theo từng quan hệ). Truy vấn và đáp án vẫn sinh từ KG thật.
- `shuffled_links`: xáo nhãn món của ảnh train (cạnh `mô_tả` sai).

In [ ]:
GRID = dict(alpha=[round(a, 2) for a in np.linspace(0, 1, 11)], lam=[0, 0.25, 0.5, 0.75, 1.0],
            eta=[0, 0.25, 0.5, 0.75, 1.0], T=[0.03, 0.1, 0.3], w2=[0.5, 1.0])

def run(model_key, lang, variant, seed, qfield=None, keep_parts=False):
    qfield = qfield or lang
    I, T = FEATS[model_key]
    split = SPLITS[seed]
    labels = IMG_DISH.copy()
    if variant == "shuffled_links":
        rng = np.random.default_rng(900 + seed)
        tr = np.array([IIDX[i] for i in IMG_IDS if split[i] == "train"])
        labels[tr] = rng.permutation(labels[tr])
    triples = variant_triples(variant, seed)
    R1, R2, _ = reach_mats(triples)
    ent_names = [e["name"] if lang == "vi" else kg_data.EN_ENTITY.get(e["name"], kg_data.EN_NAME.get(e["id"][5:], e["name"]))
                 for e in ents]
    ent_feat = np.stack([T[TIDX[n]] if n in TIDX else np.zeros(T.shape[1], np.float32) for n in ent_names])
    if variant == "no_kg_attr":
        desc = [ENT[d]["name"] if lang == "vi" else kg_data.EN_NAME[d[5:]] for d in DISHES]
    else:
        desc = [dish_desc(d, lang, triples) for d in DISHES]
    desc_emb = np.stack([T[TIDX[s]] if s in TIDX else T[TIDX[ENT[d]["name"] if lang == "vi" else kg_data.EN_NAME[d[5:]]]]
                         for s, d in zip(desc, DISHES)])
    model, X, E_t, dn, tau, gacc = train_gnn(I, ent_feat, triples, split, labels, seed)
    P_gnn = gnn_probs(model, X, E_t, dn, tau)
    P_knn = knn_probs(I, split, labels)
    res = {}
    for part in ("val", "test"):
        gal = [IIDX[i] for i in IMG_IDS if split[i] == part]
        gal_set = set(gal)
        qs = [q for q in QUERIES if q["use"] in (part, "both") and qfield in q]
        gold = [{gal.index(k) for k in gal if DISHES[IMG_DISH[k]] in q["gold"]} for q in qs]
        keep = [n for n, g in enumerate(gold) if g]
        qs = [qs[n] for n in keep]; gold = [gold[n] for n in keep]
        texts = [q[qfield] for q in qs]
        qv = np.stack([T[TIDX[t]] for t in texts])
        S_clip = qv @ I[gal].T
        Skg_by_w2 = {w2: query_dish_scores(texts, lang, R1, R2, w2, desc_emb, T) for w2 in GRID["w2"]}
        G = np.zeros((len(qs), len(gal)), bool)
        for n, g in enumerate(gold):
            G[n, list(g)] = True
        res[part] = dict(qs=qs, gold=gold, gal=gal, S_clip=S_clip, Skg=Skg_by_w2, G=G)
    def score(part, a, lam, eta, Tq, w2):
        r = res[part]
        Skg, Stxt, _ = r["Skg"][w2]
        Q = softmax(eta * Skg + (1 - eta) * zrow(Stxt) * 0.1, Tq)
        Pimg = lam * P_gnn[r["gal"]] + (1 - lam) * P_knn[r["gal"]]
        S_path = Q @ Pimg.T
        return a * zrow(r["S_clip"]) + (1 - a) * zrow(S_path), Q, Pimg
    # chọn tham số trên val
    best = None
    for w2, eta, Tq, lam in itertools.product(GRID["w2"], GRID["eta"], GRID["T"], GRID["lam"]):
        for a in GRID["alpha"]:
            S, _, _ = score("val", a, lam, eta, Tq, w2)
            m = fast_mrr(S, res["val"]["G"])
            if best is None or m > best[0] + 1e-9:
                best = (m, dict(alpha=a, lam=lam, eta=eta, T=Tq, w2=w2))
    hp = best[1]
    S_test, Q_test, Pimg_test = score("test", **{"a": hp["alpha"], "lam": hp["lam"], "eta": hp["eta"], "Tq": hp["T"], "w2": hp["w2"]})
    ev_full = evaluate(S_test, res["test"]["gold"])
    ev_clip = evaluate(res["test"]["S_clip"], res["test"]["gold"])
    meta = pd.DataFrame([dict(qid=q["qid"], group=q["group"], hops=q["hops"]) for q in res["test"]["qs"]])
    out = dict(hp=hp, val_mrr=best[0], gnn_val_acc=gacc,
               full=pd.concat([meta, ev_full], axis=1), clip=pd.concat([meta, ev_clip], axis=1))
    if keep_parts:
        out.update(S_test=S_test, Q_test=Q_test, P_gnn=P_gnn, P_knn=P_knn, res=res, labels=labels,
                   gnn=(model, X, E_t, dn, tau), triples=triples, desc_emb=desc_emb, I=I, T=T)
    return out

## 6. Thí nghiệm chính và ablation (mô hình chính, truy vấn tiếng Việt có dấu)

In [ ]:
def summarize(rows, label, by_group=True):
    df = pd.concat(rows)
    recs = []
    for g in (["all", "direct", "indirect", "paraphrase"] if by_group else ["all"]):
        d = df if g == "all" else df[df.group == g]
        if len(d) == 0:
            continue
        per_seed = d.groupby("seed")[["R1", "R5", "R10", "MRR", "nDCG10"]].mean()
        rec = dict(config=label, group=g, n_queries=int(len(d) / d.seed.nunique()))
        for m in ["R1", "R5", "R10", "MRR", "nDCG10"]:
            rec[m] = per_seed[m].mean(); rec[m + "_std"] = per_seed[m].std(ddof=0)
        recs.append(rec)
    return recs

RUNS = {}
t0 = time.time()
for variant in ["full", "no_kg_attr", "random_kg", "shuffled_links"]:
    for s in SEEDS:
        RUNS[(MAIN, "vi", variant, s)] = run(MAIN, "vi", variant, s, keep_parts=(variant == "full" and s == 0))
    print(f"{variant}: xong ({time.time() - t0:.0f}s); tham số seed 0: {RUNS[(MAIN, 'vi', variant, 0)]['hp']}, "
          f"GNN val acc {RUNS[(MAIN, 'vi', variant, 0)]['gnn_val_acc']:.3f}")

def tag(df, s):
    df = df.copy(); df["seed"] = s; return df

recs = summarize([tag(RUNS[(MAIN, "vi", "full", s)]["clip"], s) for s in SEEDS], "CLIP (đóng băng)")
names = {"full": "CLIP + KG (đầy đủ)", "no_kg_attr": "Bỏ thuộc tính KG", "random_kg": "KG ngẫu nhiên", "shuffled_links": "Xáo liên kết ảnh–món"}
for v, lab in names.items():
    recs += summarize([tag(RUNS[(MAIN, "vi", v, s)]["full"], s) for s in SEEDS], lab)
MAIN_TABLE = pd.DataFrame(recs)
MAIN_TABLE.to_csv(os.path.join(OUT, "main_results.csv"), index=False)

def show(tab):
    t = tab.copy()
    for m in ["R1", "R5", "R10", "MRR", "nDCG10"]:
        t[m] = [f"{a:.3f} ± {b:.3f}" for a, b in zip(t[m], t[m + "_std"])]
    return t[["config", "group", "n_queries", "R1", "R5", "R10", "MRR", "nDCG10"]]

print(f"Mô hình: {MODELS[MAIN]}; test; trung bình ± độ lệch chuẩn qua {len(SEEDS)} seed")
show(MAIN_TABLE)

### Kiểm định cặp (bootstrap theo truy vấn)

Lấy trung bình RR của mỗi truy vấn qua 3 seed, rồi bootstrap 10.000 lần trên tập truy vấn test.

In [ ]:
def per_query_rr(variant, which="full"):
    dfs = []
    for s in SEEDS:
        d = RUNS[(MAIN, "vi", variant, s)][which][["qid", "group", "MRR"]].copy(); d["seed"] = s; dfs.append(d)
    return pd.concat(dfs).groupby(["qid", "group"])["MRR"].mean().reset_index()

def paired(a, b, n=10000, seed=0):
    m = a.merge(b, on=["qid", "group"], suffixes=("_a", "_b"))
    rng = np.random.default_rng(seed)
    out = []
    for g in ["all", "direct", "indirect", "paraphrase"]:
        d = m if g == "all" else m[m.group == g]
        if len(d) == 0:
            continue
        diff = (d.MRR_a - d.MRR_b).values
        bs = rng.choice(diff, (n, len(diff))).mean(1)
        out.append(dict(group=g, n=len(diff), delta=diff.mean(), ci_lo=np.percentile(bs, 2.5), ci_hi=np.percentile(bs, 97.5),
                        p_one_sided=(bs <= 0).mean()))
    return pd.DataFrame(out)

full_rr = per_query_rr("full")
tests = []
for lab, (a, b) in {"Đầy đủ − CLIP": (full_rr, per_query_rr("full", "clip")),
                    "Đầy đủ − bỏ thuộc tính KG": (full_rr, per_query_rr("no_kg_attr")),
                    "Đầy đủ − KG ngẫu nhiên": (full_rr, per_query_rr("random_kg")),
                    "Đầy đủ − xáo liên kết ảnh": (full_rr, per_query_rr("shuffled_links"))}.items():
    t = paired(a, b); t.insert(0, "so_sánh", lab); tests.append(t)
PAIRED = pd.concat(tests)
PAIRED.to_csv(os.path.join(OUT, "paired_tests.csv"), index=False)
PAIRED.round(4)

### Đóng góp của từng thành phần (seed 0, giữ tham số đã chọn, tắt từng phần)

In [ ]:
r0 = RUNS[(MAIN, "vi", "full", 0)]
hp0 = r0["hp"]
def score_with(r, a, lam, eta, Tq, w2):
    R = r["res"]["test"]
    Skg, Stxt, _ = R["Skg"][w2]
    Q = softmax(eta * Skg + (1 - eta) * zrow(Stxt) * 0.1, Tq)
    Pimg = lam * r["P_gnn"][R["gal"]] + (1 - lam) * r["P_knn"][R["gal"]]
    return a * zrow(R["S_clip"]) + (1 - a) * zrow(Q @ Pimg.T)
abl = []
cfgs = {"Tham số chọn trên val": hp0,
        "Chỉ đồ thị (α = 0)": dict(hp0, alpha=0.0),
        "Chỉ CLIP (α = 1)": dict(hp0, alpha=1.0),
        "Ảnh→món chỉ bằng kNN (λ = 0)": dict(hp0, lam=0.0),
        "Ảnh→món chỉ bằng GNN (λ = 1)": dict(hp0, lam=1.0),
        "Truy vấn→món chỉ ràng buộc KG (η = 1)": dict(hp0, eta=1.0),
        "Truy vấn→món chỉ mô tả node (η = 0)": dict(hp0, eta=0.0)}
for lab, h in cfgs.items():
    ev = evaluate(score_with(r0, h["alpha"], h["lam"], h["eta"], h["T"], h["w2"]), r0["res"]["test"]["gold"])
    meta = pd.DataFrame([dict(group=q["group"]) for q in r0["res"]["test"]["qs"]])
    ev = pd.concat([meta, ev], axis=1)
    abl.append(dict(cấu_hình=lab, MRR_all=ev.MRR.mean(), MRR_direct=ev[ev.group == "direct"].MRR.mean(),
                    MRR_indirect=ev[ev.group == "indirect"].MRR.mean(),
                    MRR_paraphrase=ev[ev.group == "paraphrase"].MRR.mean(), R1_all=ev.R1.mean()))
COMP = pd.DataFrame(abl)
COMP.to_csv(os.path.join(OUT, "component_ablation_seed0.csv"), index=False)
print("Tham số seed 0:", hp0)
COMP.round(3)

## 7. Giải thích (xAI): fidelity, validity, sparsity

Với mỗi truy vấn test và ảnh top-1, giải thích gồm:
1. **Món trung gian** $d^*$ đóng góp nhiều nhất vào điểm đồ thị ($P(d\mid q)\,p(d\mid i)$).
2. **Đường đi trong KG** nối thực thể được nhắc trong truy vấn với $d^*$ (1 hoặc 2 bước), hoặc "truy vấn nhắc tên món".
3. **Bằng chứng ảnh**: tối đa 3 ảnh train gần nhất có cạnh `mô_tả` → $d^*$.

**Fidelity**: xoá đúng các cạnh trong giải thích (triple KG trên đường đi + cạnh `mô_tả` của ảnh bằng chứng) rồi chấm lại, so với xoá cùng số cạnh ngẫu nhiên (trung bình 5 lần). Mô hình không huấn luyện lại; mô tả văn bản của node giữ nguyên nên con số này là cận dưới.
**Validity**: $d^*$ có đúng là món của ảnh top-1 không; mọi triple trong giải thích có thật trong KG không.
**Sparsity**: số cạnh trong giải thích; tỉ lệ điểm đồ thị do $d^*$ đóng góp.

In [ ]:
def explain(r, qn, img_k, hp):
    """Trả về giải thích cho truy vấn thứ qn và ảnh (chỉ số toàn cục) img_k."""
    R = r["res"]["test"]; q = R["qs"][qn]
    gpos = R["gal"].index(img_k)
    Skg, Stxt, links = R["Skg"][hp["w2"]]
    Q = softmax(hp["eta"] * Skg + (1 - hp["eta"]) * zrow(Stxt) * 0.1, hp["T"])
    Pimg = hp["lam"] * r["P_gnn"][img_k] + (1 - hp["lam"]) * r["P_knn"][img_k]
    contrib = Q[qn] * Pimg
    d_star = DISHES[int(contrib.argmax())]
    paths = []
    for e in links[qn]:
        if e == d_star:
            paths.append([("truy_vấn", "nhắc_tên", d_star)])
        elif e in DIDX:
            continue
        elif any(h == d_star and t == e for h, _, t in kg_triples):
            rel = [rr for h, rr, t in kg_triples if h == d_star and t == e][0]
            paths.append([(d_star, rel, e)])
        elif (d_star, e) in PATH2_true:
            r1, mid, r2 = PATH2_true[(d_star, e)][0]
            paths.append([(d_star, r1, mid), (mid, r2, e)])
    tr = np.array([IIDX[i] for i in IMG_IDS if SPLITS[0][i] == "train"])
    I = r["I"]
    cand = [k for k in tr[np.argsort(-(I[tr] @ I[img_k]))] if r["labels"][k] == DIDX[d_star]][:3]
    return dict(q=q, img=img_k, d_star=d_star, share=float(contrib.max() / (contrib.sum() + 1e-12)),
                paths=paths, evidence=cand, contrib=contrib)

def rescore_without(r, qn, img_k, hp, drop_triples, drop_links):
    """Chấm lại ảnh img_k cho truy vấn qn khi bỏ một số triple KG và cạnh mô_tả (không huấn luyện lại)."""
    R = r["res"]["test"]; q = R["qs"][qn]
    trip = [t for t in r["triples"] if t not in drop_triples]
    R1, R2, _ = reach_mats(trip)
    Skg, Stxt, _ = query_dish_scores([q["vi"]], "vi", R1, R2, hp["w2"], r["desc_emb"], r["T"])
    Q = softmax(hp["eta"] * Skg + (1 - hp["eta"]) * zrow(Stxt) * 0.1, hp["T"])[0]
    labels = r["labels"].copy()
    split = dict(SPLITS[0])
    for k in drop_links:
        split[IMG_IDS[k]] = "removed"
    P_knn = knn_probs(r["I"], split, labels)
    model, X, E_t, dn, tau = r["gnn"]
    n_img = len(IMG_IDS)
    dl = set(drop_links)
    E2 = []
    for (s, d) in E_t:
        s_, d_ = s.numpy(), d.numpy()
        keepm = ~(np.isin(s_, list(dl)) & (d_ >= n_img)) & ~(np.isin(d_, list(dl)) & (s_ >= n_img))
        E2.append((s[torch.tensor(keepm)], d[torch.tensor(keepm)]))
    P_gnn = gnn_probs(model, X, E2, dn, tau)
    gal = R["gal"]
    Pimg = hp["lam"] * P_gnn[gal] + (1 - hp["lam"]) * P_knn[gal]
    S = hp["alpha"] * zrow(R["S_clip"][qn:qn + 1]) + (1 - hp["alpha"]) * zrow((Q[None] @ Pimg.T))
    S0 = r["S_test"][qn]
    gpos = gal.index(img_k)
    return float(S0[gpos] - S[0, gpos]), int((S[0] > S[0, gpos]).sum() + 1), int((S0 > S0[gpos]).sum() + 1)

rng = random.Random(0)
XAI = []
kg_attr_triples = [t for t in kg_triples if t[1] in ATTR_RELS + LINK_RELS]
tr_idx = [IIDX[i] for i in IMG_IDS if SPLITS[0][i] == "train"]
for qn, q in enumerate(r0["res"]["test"]["qs"]):
    gal = r0["res"]["test"]["gal"]
    top = gal[int(np.argmax(r0["S_test"][qn]))]
    ex = explain(r0, qn, top, hp0)
    tri = [t for p in ex["paths"] for t in p if t[0] != "truy_vấn"]
    links = ex["evidence"]
    n_edges = len(tri) + len(links) + sum(1 for p in ex["paths"] for t in p if t[0] == "truy_vấn")
    drop_e, rank_after, rank_before = rescore_without(r0, qn, top, hp0, set(tri), links)
    rnd = []
    for _ in range(5):
        rt = set(rng.sample(kg_attr_triples, len(tri)))
        rl = rng.sample(tr_idx, len(links))
        rnd.append(rescore_without(r0, qn, top, hp0, rt, rl))
    XAI.append(dict(qid=q["qid"], group=q["group"], query=q["vi"], top1_image=IMG_IDS[top],
                    top1_dish=ENT[DISHES[IMG_DISH[top]]]["name"], top1_correct=DISHES[IMG_DISH[top]] in q["gold"],
                    d_star=ENT[ex["d_star"]]["name"], valid_dish=ex["d_star"] == DISHES[IMG_DISH[top]],
                    triples_exist=all(t in set(kg_triples) for t in tri),
                    n_explanation_edges=n_edges, share_dstar=ex["share"],
                    drop_expl=drop_e, drop_rand=float(np.mean([x[0] for x in rnd])),
                    rank_before=rank_before, rank_after_expl=rank_after,
                    rank_after_rand=float(np.mean([x[1] for x in rnd])),
                    explanation=" | ".join(" → ".join(f"{ENT[h]['name'] if h in ENT else h} —{rr}→ {ENT[t]['name'] if t in ENT else t}" for h, rr, t in p) for p in ex["paths"])
                                 + (f" | ảnh bằng chứng: {', '.join(IMG_IDS[k] for k in links)}" if links else "")))
XAI = pd.DataFrame(XAI)
XAI.to_csv(os.path.join(OUT, "xai_per_query.csv"), index=False)
summ = dict(n_queries=len(XAI),
            fidelity_score_drop_expl=XAI.drop_expl.mean(), fidelity_score_drop_random=XAI.drop_rand.mean(),
            frac_rank_worse_expl=(XAI.rank_after_expl > XAI.rank_before).mean(),
            frac_rank_worse_random=(XAI.rank_after_rand > XAI.rank_before).mean(),
            validity_dstar_is_image_dish=XAI.valid_dish.mean(), validity_triples_exist=XAI.triples_exist.mean(),
            sparsity_mean_edges=XAI.n_explanation_edges.mean(), share_of_graph_score_by_dstar=XAI.share_dstar.mean(),
            top1_correct=XAI.top1_correct.mean())
json.dump(summ, open(os.path.join(OUT, "xai_summary.json"), "w"), indent=1)
for k, v in summ.items():
    print(f"{k:38s} {v:.3f}" if isinstance(v, float) else f"{k:38s} {v}")
print("\nTheo nhóm truy vấn:")
print(XAI.groupby("group")[["drop_expl", "drop_rand", "valid_dish", "n_explanation_edges"]].mean().round(3))

In [ ]:
pd.set_option("display.max_colwidth", 160)
XAI[["group", "query", "top1_dish", "top1_correct", "d_star", "explanation"]].head(12)

## 8. Phân tích khả năng chuyển giao

Câu hỏi: phương pháp xây cho Flickr8k / Action Genome (tiếng Anh, đồ thị cảnh) có chuyển sang MMKG tiếng Việt về ẩm thực không, và phần nào chuyển được?

Chạy cùng quy trình (chọn tham số trên val, 3 seed) với:
- CLIP đa ngữ nhỏ (`xlm-roberta-base-ViT-B-32`), truy vấn tiếng Việt.
- CLIP tiếng Anh (`ViT-L-14`, backbone của phần A) với truy vấn tiếng Việt, và với truy vấn tiếng Anh (bản dịch sinh từ KG).
- CLIP đa ngữ chính với truy vấn tiếng Việt **không dấu** và với truy vấn tiếng Anh.

In [ ]:
TRANSFER = []
setups = [("mclip_b32", "vi", "vi", "CLIP đa ngữ B/32, tiếng Việt"),
          ("clip_l14_en", "vi", "vi", "CLIP tiếng Anh L/14, truy vấn tiếng Việt"),
          ("clip_l14_en", "en", "en", "CLIP tiếng Anh L/14, truy vấn tiếng Anh"),
          (MAIN, "vi", "vi_nodiac", "CLIP đa ngữ chính, tiếng Việt không dấu"),
          (MAIN, "en", "en", "CLIP đa ngữ chính, truy vấn tiếng Anh")]
TRANSFER += [dict(r, setup=f"{MODELS[MAIN][0]}, tiếng Việt có dấu") for r in
             summarize([tag(RUNS[(MAIN, "vi", "full", s)]["clip"], s) for s in SEEDS], "CLIP") +
             summarize([tag(RUNS[(MAIN, "vi", "full", s)]["full"], s) for s in SEEDS], "CLIP + KG")]
for mk, lang, qf, lab in setups:
    if mk not in FEATS:
        print("Bỏ qua (không có đặc trưng):", lab); continue
    rs = {s: run(mk, lang, "full", s, qfield=qf) for s in SEEDS}
    for s in SEEDS:
        RUNS[(mk, lang, "full", s, qf)] = rs[s]
    TRANSFER += [dict(r, setup=lab) for r in summarize([tag(rs[s]["clip"], s) for s in SEEDS], "CLIP") +
                 summarize([tag(rs[s]["full"], s) for s in SEEDS], "CLIP + KG")]
TR = pd.DataFrame(TRANSFER)
TR.to_csv(os.path.join(OUT, "transfer_results.csv"), index=False)
piv = TR.pivot_table(index=["setup", "group"], columns="config", values="MRR").reset_index()
piv["ΔMRR"] = piv["CLIP + KG"] - piv["CLIP"]
piv.round(3)

In [ ]:
mm = MAIN_TABLE[(MAIN_TABLE.group == "all")].set_index("config")["MRR"]
mm_i = MAIN_TABLE[(MAIN_TABLE.group == "indirect")].set_index("config")["MRR"]
mm_d = MAIN_TABLE[(MAIN_TABLE.group == "direct")].set_index("config")["MRR"]
mm_p = MAIN_TABLE[(MAIN_TABLE.group == "paraphrase")].set_index("config")["MRR"]
cross = pd.DataFrame([
    dict(tập="Flickr8k (phần A, ViT-L/14)", CLIP=0.800, **{"CLIP + đồ thị": 0.833}, ghi_chú="caption tiếng Anh; đồ thị ít thông tin mới"),
    dict(tập="Action Genome (phần A, ViT-L/14)", CLIP=0.171, **{"CLIP + đồ thị": 0.650}, ghi_chú="nhãn hành động/bối cảnh mà CLIP không thấy"),
    dict(tập="MMKG ẩm thực — truy vấn trực tiếp", CLIP=mm_d["CLIP (đóng băng)"], **{"CLIP + đồ thị": mm_d["CLIP + KG (đầy đủ)"]}, ghi_chú="tên món tiếng Việt"),
    dict(tập="MMKG ẩm thực — truy vấn gián tiếp", CLIP=mm_i["CLIP (đóng băng)"], **{"CLIP + đồ thị": mm_i["CLIP + KG (đầy đủ)"]}, ghi_chú="thuộc tính chỉ có trong KG"),
    dict(tập="MMKG ẩm thực — diễn đạt tự nhiên", CLIP=mm_p["CLIP (đóng băng)"], **{"CLIP + đồ thị": mm_p["CLIP + KG (đầy đủ)"]}, ghi_chú="viết tay, từ ngữ ngoài KG"),
])
cross["Δ"] = cross["CLIP + đồ thị"] - cross["CLIP"]
cross.to_csv(os.path.join(OUT, "cross_dataset.csv"), index=False)
cross.round(3)

## 9. Demo: nhập truy vấn → top-k ảnh + giải thích

In [ ]:
def search(query, k=5, lang="vi", show_images=True):
    """Truy vấn trên ảnh test của seed 0 bằng mô hình đầy đủ, in đường đi giải thích."""
    r = r0; hp = hp0; I, T = FEATS[MAIN]
    R = r["res"]["test"]; gal = R["gal"]
    if query in TIDX:
        qv = T[TIDX[query]]
    elif SMOKE:
        qv = np.random.default_rng(0).normal(size=T.shape[1]).astype(np.float32); qv /= np.linalg.norm(qv)
    else:
        import torch, open_clip
        arch, pt = MODELS[MAIN]
        if "_demo" not in globals():
            m, _, _ = open_clip.create_model_and_transforms(arch, pretrained=pt)
            globals()["_demo"] = (m.eval(), open_clip.get_tokenizer(arch))
        m, tk = globals()["_demo"]
        with torch.no_grad():
            qv = F.normalize(m.encode_text(tokenize(tk, [query])).float(), dim=-1)[0].numpy()
    R1, R2, _ = reach_mats(kg_triples)
    es = link_entities(query, lang)
    Skg = np.zeros((1, len(DISHES)), np.float32)
    cons = [e for e in es if e not in DIDX]
    if cons:
        Skg[0] = np.stack([np.maximum(R1[:, EIDX[e]], hp["w2"] * R2[:, EIDX[e]]) for e in cons]).mean(0)
    for e in es:
        if e in DIDX: Skg[0, DIDX[e]] = 1
    Stxt = qv[None] @ r["desc_emb"].T
    Q = softmax(hp["eta"] * Skg + (1 - hp["eta"]) * zrow(Stxt) * 0.1, hp["T"])
    Pimg = hp["lam"] * r["P_gnn"][gal] + (1 - hp["lam"]) * r["P_knn"][gal]
    S = hp["alpha"] * zrow(qv[None] @ I[gal].T) + (1 - hp["alpha"]) * zrow(Q @ Pimg.T)
    order = np.argsort(-S[0])[:k]
    print(f"Truy vấn: {query!r}\nThực thể nhận ra: {[ENT[e]['name'] for e in es]}")
    print("Món ứng viên từ KG:", ", ".join(f"{ENT[DISHES[j]]['name']} ({Q[0, j]:.2f})" for j in np.argsort(-Q[0])[:3]))
    rows = []
    for rank, gp in enumerate(order, 1):
        kimg = gal[gp]
        contrib = Q[0] * Pimg[gp]
        ds = DISHES[int(contrib.argmax())]
        why = []
        for e in es:
            if e == ds: why.append(f"truy vấn nhắc tên «{ENT[ds]['name']}»")
            elif e in DIDX: continue
            elif R1[DIDX[ds], EIDX[e]]:
                rel = [rr for h, rr, t in kg_triples if h == ds and t == e][0]
                why.append(f"{ENT[ds]['name']} —{rel}→ {ENT[e]['name']}")
            elif (ds, e) in PATH2_true:
                r1_, mid, r2_ = PATH2_true[(ds, e)][0]
                why.append(f"{ENT[ds]['name']} —{r1_}→ {ENT[mid]['name']} —{r2_}→ {ENT[e]['name']}")
        print(f" {rank}. {IMG_IDS[kimg]:28s} điểm {S[0, gp]:+.2f} | ảnh là «{ENT[DISHES[IMG_DISH[kimg]]]['name']}»"
              f" | qua món «{ENT[ds]['name']}» ({contrib.max() / contrib.sum():.0%} điểm đồ thị): {'; '.join(why) or 'không có ràng buộc KG khớp'}")
        rows.append(kimg)
    if show_images and not SMOKE:
        import matplotlib.pyplot as plt
        from PIL import Image
        fig, ax = plt.subplots(1, len(rows), figsize=(3 * len(rows), 3))
        for a, kimg in zip(np.atleast_1d(ax), rows):
            a.imshow(Image.open(imgs.path[kimg])); a.axis("off"); a.set_title(ENT[DISHES[IMG_DISH[kimg]]]["name"], fontsize=9)
        plt.tight_layout(); plt.show()

search("bánh mặn của Huế, có tôm")

In [ ]:
search("món sợi nước của Nam Bộ, có mắm cá")

In [ ]:
search("đồ uống của Hà Nội có trứng")

## 10. Tóm tắt

In [ ]:
full_all = MAIN_TABLE[(MAIN_TABLE.config == "CLIP + KG (đầy đủ)") & (MAIN_TABLE.group == "all")].iloc[0]
clip_all = MAIN_TABLE[(MAIN_TABLE.config == "CLIP (đóng băng)") & (MAIN_TABLE.group == "all")].iloc[0]
summary = dict(
    model=MODELS[MAIN], seeds=SEEDS, smoke=SMOKE,
    kg=dict(entities=len(ents), triples_total=len(all_triples), triples_knowledge=len(kg_triples),
            triples_image=len(img_triples), images=len(imgs), dishes=len(DISHES)),
    queries_test=dict(Counter(q["group"] for q in r0["res"]["test"]["qs"])),
    test_all=dict(CLIP_MRR=clip_all.MRR, FULL_MRR=full_all.MRR, CLIP_R1=clip_all.R1, FULL_R1=full_all.R1),
    by_group={g: {c: float(MAIN_TABLE[(MAIN_TABLE.config == c) & (MAIN_TABLE.group == g)].MRR.iloc[0]) for c in MAIN_TABLE.config.unique()}
              for g in ["direct", "indirect", "paraphrase"]},
    paired=PAIRED.to_dict("records"), xai=summ, hparams_seed0=hp0)
json.dump(summary, open(os.path.join(OUT, "summary.json"), "w"), indent=1, ensure_ascii=False, default=float)
print(json.dumps(summary["kg"], ensure_ascii=False))
print(f"Test (tất cả truy vấn): CLIP MRR {clip_all.MRR:.3f} → CLIP + KG {full_all.MRR:.3f}; R@1 {clip_all.R1:.3f} → {full_all.R1:.3f}")
for g in ["direct", "indirect", "paraphrase"]:
    print(g, {k: round(v, 3) for k, v in summary["by_group"][g].items()})